# SMM Final Project — Colab GPU Runner

跨國 zero-shot 實驗的 Colab 執行簿本。用 GPU 跑 TSET / DMC / AMCv2 / **CS-DFA（方向三）**。

**工作流**：
- **data**（~3.6GB）放 Google Drive，一次上傳長期重用。
- **code** 每次從 GitHub fork `git clone` 到 Colab 本地（第 3.5 節），永遠跑最新版。

**使用前提**：Drive 裡 `PROJECT_DIR` 底下要有 `data/processed/<country>/`。code 不需事先放 Drive。

**建議畫面設定**：Runtime → Change runtime type → Hardware accelerator 選 **T4 GPU**（或更好）。


## 1. 確認 GPU 已啟用

In [ ]:
!nvidia-smi

## 2. 挂載 Google Drive

授權後，你的 Drive 會出現在 `/content/drive/MyDrive/`。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. 設定專案路徑

把 `PROJECT_DIR` 改成你 Drive 裡放專案的位置。該路徑底下要有 `src/` 與 `data/processed/`。

In [ ]:
import os

# ← 改成你的實際路徑（Drive 裡放 data 的位置；code 由下一節 git clone，不需放這）
PROJECT_DIR = '/content/drive/MyDrive/SMM_Final_Project'

assert os.path.isdir(os.path.join(PROJECT_DIR, 'data', 'processed')), \
    f'找不到 data/processed/，請確認資料已上傳到: {PROJECT_DIR}'
print('資料路徑 OK')
print('國家資料夾:', sorted(os.listdir(os.path.join(PROJECT_DIR, 'data', 'processed'))))


### （重要）從 GitHub fork 取最新 code（data 留在 Drive）

把 **code（git）** 跟 **data（Drive）** 拆開：每次從 fork clone 最新 `src/` 到 Colab 本地，
`data/` 用 symlink 指回 Drive（太大不複製），`results/` 也 symlink 回 Drive 保留 log。

以下預設 branch 指向目前整合過 **TSET runner / CS-DFA / results 分流** 的版本；若你之後改在別的 branch 開發，只要改 `BRANCH` 再重跑這格即可。

In [ ]:
%cd /content
import os, shutil

REPO_URL  = 'https://github.com/NelsonKCT/SMM_Final_Project.git'
BRANCH    = 'nelson/cs-dfa'
LOCAL_DIR = '/content/smm'

# 若 repo 是 private，改用帶 token 的 URL（PAT 在 GitHub Settings -> Developer settings 產生）：
# REPO_URL = 'https://<GITHUB帳號>:<PAT>@github.com/NelsonKCT/SMM_Final_Project.git'

# 1) 砍掉舊的，clone 最新 code 到本地
shutil.rmtree(LOCAL_DIR, ignore_errors=True)
!git clone -b {BRANCH} {REPO_URL} {LOCAL_DIR}

# clone 失敗就直接停，避免後面 symlink 把問題蓋掉
assert os.path.isdir(f'{LOCAL_DIR}/src'), 'clone 失敗，先看上面 git 的輸出'

# 2) data 太大不複製，symlink 指回 Drive（只讀）
if os.path.lexists(f'{LOCAL_DIR}/data'):
    (os.remove if os.path.islink(f'{LOCAL_DIR}/data') else shutil.rmtree)(f'{LOCAL_DIR}/data')
os.symlink(f'{PROJECT_DIR}/data', f'{LOCAL_DIR}/data')

# 3) results symlink 回 Drive，斷線也能保留已跑完的 log
for subdir in ['results', 'results/tset_logs', 'results/csdfa_logs', 'results/diagnostics']:
    os.makedirs(f'{PROJECT_DIR}/{subdir}', exist_ok=True)
if os.path.lexists(f'{LOCAL_DIR}/results'):
    (os.remove if os.path.islink(f'{LOCAL_DIR}/results') else shutil.rmtree)(f'{LOCAL_DIR}/results')
os.symlink(f'{PROJECT_DIR}/results', f'{LOCAL_DIR}/results')

RUN_DIR = LOCAL_DIR

# 4) 驗證：抓到的 commit + 關鍵檔案/資料夾是否存在
print('\n--- 驗證 ---')
!cd {RUN_DIR} && git log --oneline -1
print('run_experiments 存在：', os.path.exists(f'{RUN_DIR}/src/run_experiments.py'))
print('CSDFA 檔存在：', os.path.exists(f'{RUN_DIR}/src/run_MultiModalGNN_CrossAttention_CrossCountry_CSDFA.py'))
print('tset_logs dir：', os.path.isdir(f'{PROJECT_DIR}/results/tset_logs'))
print('csdfa_logs dir：', os.path.isdir(f'{PROJECT_DIR}/results/csdfa_logs'))
print('RUN_DIR:', RUN_DIR, '->', sorted(os.listdir(RUN_DIR)))


## 4. 安裝依賴

Colab 已預裝 torch（帶 CUDA）。這裡裝 torch_geometric 與專案用到的其他套件。
torch_geometric 的 GCN/SAGE/GAT 在新版不需要編譯版 torch_scatter。

In [ ]:
import torch
print('torch', torch.__version__, 'cuda available:', torch.cuda.is_available())

# 只裝 torch_geometric 與 mlflow。
# 不要裝 node2vec：它會拉進 gensim 把 numpy 降到 <2，與 Colab 的 numpy 2.x
# 套件 binary 不相容（numpy.dtype size changed）。TSET/DMC/AMCv2 不需要 node2vec，
# my_utils.py 已把該 import 改成 lazy（只有跑 Node2Vec baseline 才會用到）。
!pip install -q torch_geometric mlflow
# networkx / scikit-learn / scipy / pandas Colab 已有，不用裝

import torch_geometric, mlflow, networkx, sklearn
print('torch_geometric', torch_geometric.__version__, '— 依賴 OK')

## 5. Smoke test：先單跑一國確認 GPU pipeline 通

`--device 0` 會自動使用 GPU（`setup_env` 偵測 `cuda.is_available()`）。
先用 **CS-DFA 跑 iran**（資料較小）驗證 pipeline 與新方法都通。


In [ ]:
%cd {RUN_DIR}/src
!python run_experiments.py --method csdfa --countries iran --device 0


## 6. 正式批次：六國 × 三方法

一次跑一個方法，結果即時寫回 Drive。
- TSET 會寫到 `results/tset_logs/`
- CS-DFA 系列會寫到 `results/csdfa_logs/`
- diagnostics 會寫到 `results/diagnostics/`

免費版可能跑不完整批，建議分個 cell 跑；若中途斷線，可用後面的「補跑剩餘國家」cell 接著跑。

In [ ]:
%cd {RUN_DIR}/src
!python run_experiments.py --method tset --device 0

In [ ]:
%cd {RUN_DIR}/src
!python run_experiments.py --method dmc --device 0

In [ ]:
%cd {RUN_DIR}/src
!python run_experiments.py --method amcv2 --device 0

## 7. 確認結果

TSET logs 會寫在 Drive 的 `results/tset_logs/`。此處快速抽出每國的最終 test `f1_macro`。

In [ ]:
import glob, re

results_dir = f'{PROJECT_DIR}/results/tset_logs'
print(f"{'country':11s} {'SOTM':6s} {'min_MMD':9s} {'TEST_f1':8s} {'coRT_f1':8s} {'coURL_f1':8s}")
for f in sorted(glob.glob(f'{results_dir}/zero-shot_TSET_*.txt')):
    country = re.search(r'TSET_(\w+)\.txt', f).group(1)
    txt = open(f, encoding='utf-8', errors='replace').read()
    def grab(pat):
        m = re.search(pat, txt)
        return m.group(1) if m else '?'
    sotm   = grab(r'Structure-Only Mode \(SOTM\)\s*=\s*(\w+)')
    mmd    = grab(r'min_MMD \(linguistic distance\)\s*=\s*([\d.]+)')
    # 用 \[TEST\] 精確鎖定整體那行，避免 match 到 [TEST_coRT] / [TEST_tweetSim] 等 subnet
    f1     = grab(r'\[TEST\] f1_macro:\s*([\d.]+)')
    f1_rt  = grab(r'\[TEST_coRT\] f1_macro:\s*([\d.]+)')
    f1_url = grab(r'\[TEST_coURL\] f1_macro:\s*([\d.]+)')
    print(f"{country:11s} {sotm:6s} {mmd:9s} {f1:8s} {f1_rt:8s} {f1_url:8s}")

## 8. CS-DFA（方向三）— Channel-Specific DFA

把最強的 DFA 與 AMC/DMC 的「五 subnet 分通道」結合：
- **coverage-gated fusion**：節點只融合自己有邊的 channel，china 那些 cov 僅 3–7% 的噪音 channel 對未覆蓋節點自動歸 0。
- **coverage prior**：attention 加 `λ·log(coverage_c)`，給穩定 channel(coRT/coURL) 起跑優勢。
- **BCE loss**（不繼承害 cuba 的 Focal）。
- **channel-specific CORAL** 套在 coRT/coURL 的 z 嵌入，並同時印 text-proj CORAL 當對照，驗證 alignment 是否非 no-op。

> 前提：跟 DMC/AMCv2 一樣需要每國的 `edge_index_<subnet>.th`。下一格先確認。
> 也記得：本次新增的 `run_..._CSDFA.py` 與改過的 `run_experiments.py` 要先同步到 Drive 的 `src/`，再重跑上面的 Cell 8（複製 src 到本地）。


### 8.0 確認 edge_index 預存檔存在（沒有的話第一次會現算、很慢）

In [ ]:
import glob
for c in ['china', 'iran', 'UAE', 'cuba', 'russia', 'venezuela']:
    files = sorted(glob.glob(f'{PROJECT_DIR}/data/processed/{c}/edge_index_*.th'))
    names = [f.split('edge_index_')[-1].replace('.th', '') for f in files]
    print(f"{c:11s} {len(files)}/5  {names}")


### 8.1 Smoke test：先單跑 iran 確認 pipeline 通

In [ ]:
%cd {RUN_DIR}/src
!python run_experiments.py --method csdfa --countries iran --device 0


### 8.2 正式六國：CS-DFA 主結果

In [ ]:
%cd {RUN_DIR}/src
!python run_experiments.py --method csdfa --device 0


### 8.2b 若中途斷線，只補跑剩餘國家

長批次在 Colab 很容易中斷。下面這格是補跑模板：把 `--countries` 換成還沒完成的 target countries 即可。

In [ ]:
%cd {RUN_DIR}/src
# 範例：若前面只跑完 china / iran，這裡就補剩下的國家
!python run_experiments.py --method csdfa --countries UAE cuba russia venezuela --device 0


### 8.3 Ablation（有時間再跑）

建議優先順序：
1. `csdfa_nogate`：最關鍵的 decisive control；若這個掉很多，才表示 coverage-gated fusion 真的在起作用。
2. `csdfa_nocoral`：關掉 CORAL，證明增益來自 coverage-gated fusion 而非 alignment。
3. `csdfa_noprior`：`λ=0`，孤立 coverage prior 相對純 gating 的貢獻。


In [ ]:
%cd {RUN_DIR}/src
!python run_experiments.py --method csdfa_nocoral --device 0


In [ ]:
%cd {RUN_DIR}/src
!python run_experiments.py --method csdfa_noprior --device 0


In [ ]:
%cd {RUN_DIR}/src
!python run_experiments.py --method csdfa_nogate --device 0


### 8.4 看 CS-DFA 結果

兩個重點：
1. **china 的 subnet f1**（hashSeq/fastRT/tweetSim）有沒有回穩、整體 TEST f1 有沒有追上 DFA 平均 0.775。
2. **channel-z CORAL 是不是 no-op**：看 `CORAL[text=.. | channel_mean=..]`，若 channel 那欄明顯非 0 而 text≈0，就坐實「alignment 要搬到 channel-z 才有用」。


In [ ]:
import glob, re

results_dir = f'{PROJECT_DIR}/results/csdfa_logs'
SUBTAGS = ['[TEST]', '[TEST_coRT]', '[TEST_coURL]', '[TEST_hashSeq]', '[TEST_fastRT]', '[TEST_tweetSim]']
hdr = ['country'] + [t.strip('[]').replace('TEST_', '').replace('TEST', 'overall') for t in SUBTAGS]
print('  '.join(f'{h:9s}' for h in hdr))
for f in sorted(glob.glob(f'{results_dir}/zero-shot_CSDFA_*.txt')):
    country = re.search(r'CSDFA_(\w+)\.txt', f).group(1)
    txt = open(f, encoding='utf-8', errors='replace').read()
    row = [country]
    for tag in SUBTAGS:
        m = re.search(re.escape(tag) + r' f1_macro:\s*([\d.]+)', txt)
        row.append(m.group(1) if m else '?')
    print('  '.join(f'{v:9s}' for v in row))

# CORAL 診斷：抽每國最後一筆 text vs channel CORAL，確認 channel-z 是否非 no-op
print('\n--- CORAL (text vs channel_mean), 每國最後一筆 ---')
for f in sorted(glob.glob(f'{results_dir}/zero-shot_CSDFA_*.txt')):
    country = re.search(r'CSDFA_(\w+)\.txt', f).group(1)
    txt = open(f, encoding='utf-8', errors='replace').read()
    matches = re.findall(r'CORAL\[text=([\d.eE+-]+) \| channel_mean=([\d.eE+-]+)', txt)
    if matches:
        t, ch = matches[-1]
        print(f'{country:11s} text={float(t):.6f}  channel_mean={float(ch):.6f}')
    else:
        print(f'{country:11s} (no CORAL line found)')